## CHUNKING COMPARISON

In [4]:
import sys
import os

from pathlib import Path
from itertools import islice
sys.path.append(os.path.abspath('..'))

In [5]:
from src.retriever import VectorRetriever
from src.document_processor import load_document
from sentence_transformers import SentenceTransformer

In [6]:
# Load the model (downloads automatically on first run)
model = SentenceTransformer('all-MiniLM-L6-v2')

# Initialize the retriever with the loaded model
retriever = VectorRetriever(model=model)


retriever = VectorRetriever(model)
files = ["python.txt", "docker.txt", "database.txt", "machine_learning.txt"]

def retrieve_with_index(chunk_type):
    for filename in files:
        # 1. Use YOUR custom function directly to get the perfect list of SpaCy sentences
        # (Make sure the path matches where your raw documents actually live)
        document_data = load_document(f"data/documents/{filename}", "programming")
        sentence_list = document_data["text"] 
            
        if chunk_type == "semantic":
            chunks = retriever.semantic_chunks(sentence_list) 
            
        elif chunk_type == "sentence":
            chunks = retriever.sentence_chunks(sentence_list, max_length=200, overlap_sentences=2) 
            
        else:
            # Fixed size needs a single string, so we just join the sentences for this one case
            raw_text = " ".join(sentence_list)
            chunks = retriever.fixed_size_chunks(raw_text, 200, 20)
        
        # 3. Index with metadata
        retriever.index_chunks(chunks, source_document=filename)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5549.80it/s]


In [8]:
def evaluate_retriever(retriever, queries, k=5):
    precisions = []
    recalls = []
    reciprocal_ranks = []
    
    for item in queries:
        query = item["query"]
        expected_doc = item["expected_document"]
        
        # Search top-k results
        results = retriever.search(query, k=k)
        
        # NOTE: Make sure your VectorRetriever.search() is returning "source_document"
        retrieved_docs = [r["source_document"] for r in results]
        
        # Precision@k
        relevant_count = retrieved_docs.count(expected_doc)
        precisions.append(relevant_count / k)
        
        # Recall@k (1 if the document is in the results at all, else 0)
        recalls.append(1.0 if expected_doc in retrieved_docs else 0.0)
        
        # MRR (1 / rank of the first correct document)
        rr = 0.0
        for rank, doc in enumerate(retrieved_docs, start=1):
            if doc == expected_doc:
                rr = 1.0 / rank
                break
        reciprocal_ranks.append(rr)
        
    return {
        "precision@k": sum(precisions) / len(precisions),
        "recall@k": sum(recalls) / len(recalls),
        "mrr": sum(reciprocal_ranks) / len(reciprocal_ranks)
    }

In [ ]:
# Assuming eval_queries is already loaded from your queries.json
strategies = ["fixed", "sentence", "semantic"]
comparison_results = {}

for strat in strategies:
    # Clear the database for a fresh start each time
    retriever.chunk_dicts = []
    
    # 1. Index the documents using the selected strategy
    retrieve_with_index(strat)
    
    # 2. Evaluate
    metrics = evaluate_retriever(retriever, eval_queries, k=5)
    metrics["total_chunks"] = len(retriever.chunk_dicts)
    
    comparison_results[strat] = metrics

# 3. Print the final leaderboard
for strat, metrics in comparison_results.items():
    print(f"--- {strat.upper()} ---")
    print(f"Total Chunks: {metrics['total_chunks']}")
    print(f"Precision@5: {metrics['precision@k']:.2f}")
    print(f"Recall@5: {metrics['recall@k']:.2f}")
    print(f"MRR: {metrics['mrr']:.2f}\n")